In [3]:
import pandas as pd
import re
from collections import defaultdict

# Lista expandida de gatilhos metacognitivos com padrões mais flexíveis
gatilhos_metacognitivos = {
    'reconhecimento_erro': [
        # Percepção de erro
        "di cuenta", "me equivo", "confund", "cost[oóa]", "dificil",
        "no entend", "no comprend", "no sup", "duda", "error", "mal",
        "fall[oóeéa]", "mistake", "equivoc", "problema", "confus",
        
        # Reflexão sobre erro
        "pens[eéa]", "cre[iyí]", "parec[eéi]", "sent[ií]",
        
        # Dificuldade
        "complicado", "dificultad", "problema", "no puedo", "no pude",
        "no logr[eéo]", "no alcanz[eéo]", "no consig[oua]"
    ],
    'reflexao': [
        # Pensamento reflexivo
        "reflex", "pens[eéa]", "anal", "pregunt", "consider",
        "evalu", "revis", "comprend", "record", "entend",
        "observ", "not[eéa]", "mir[eéa]", "estudi[eéa]",
        
        # Expressões de análise
        "me parec[eéi]", "creo que", "pienso que", "supongo que",
        "me pregunto", "me cuestiono", "debo", "debería",
        
        # Tomada de consciência
        "me doy cuenta", "ahora veo", "ahora entiendo",
        "me percato", "caigo en", "noto que"
    ],
    'monitoramento': [
        # Ações de verificação
        "revis", "verific", "comprob", "repas", "volv[ií]", 
        "consult", "fij[eéa]", "control", "cheque", "mir[eéa]",
        
        # Expressões de monitoramento
        "estoy segur", "quiero ver", "necesito ver",
        "debo revisar", "tengo que ver", "voy a ver",
        
        # Busca de informação
        "busqu[eéa]", "investig", "pregunt", "le[ií]", "mir[eéa]",
        "encuentr", "hall[eéa]"
    ],
    'descoberta': [
        # Momento de descoberta
        "descubr", "sorprend", "llam[oóa] la atenci[oóa]n",
        "entend[ií]", "comprend[ií]", "me di cuenta",
        "not[eéa]", "observ[eéa]", "apareci[oóa]",
        
        # Expressões de surpresa
        "wow", "increible", "interesante", "curioso",
        "fascinante", "impresionante", "sorprendente",
        
        # Mudança de perspectiva
        "cambi[oóa]", "diferente", "otro punto", "nueva forma"
    ],
    'planejamento': [
        # Ações de planejamento
        "planific", "organiz", "decid", "cambi[eéa]", "opt[eéa]",
        "prefer", "eleg", "propu", "prepar", "establec",
        
        # Expressões de intenção
        "voy a", "tengo que", "debo", "necesito", "quiero",
        "planeo", "pienso", "pretendo",
        
        # Estratégias
        "estrategia", "manera", "forma", "método", "paso",
        "primero", "después", "luego", "finalmente"
    ]
}

def encontrar_gatilhos(texto):
    """
    Encontra gatilhos metacognitivos no texto de forma mais flexível.
    """
    if pd.isna(texto):
        return {'gatilhos': [], 'categorias': []}
    
    texto = texto.lower()
    gatilhos_encontrados = set()
    categorias = set()
    
    # Procurar por cada gatilho no texto
    for categoria, padroes in gatilhos_metacognitivos.items():
        for padrao in padroes:
            # Usando regex de forma mais flexível
            if re.search(padrao, texto, re.IGNORECASE):
                gatilhos_encontrados.add(padrao)
                categorias.add(categoria)
    
    return {
        'gatilhos': list(gatilhos_encontrados),
        'categorias': list(categorias)
    }

# Carregar e processar os dados
print("Carregando dados...")
df = pd.read_csv('../data/temp/chats_edu.csv', encoding='utf-8', sep=',')

# Contar o número de mensagens por sessão
print("Contando mensagens por sessão...")
df['message_count'] = df.groupby('session_id').cumcount() + 1

# Filtrar mensagens com mais de 4 mensagens por sessão
df = df[df['message_count'] > 4]

# Aplicar a função de busca de gatilhos
print("Processando mensagens...")
resultados = df['student_query'].apply(encontrar_gatilhos)
df['gatilhos_encontrados'] = resultados.apply(lambda x: x['gatilhos'])
df['categorias_encontradas'] = resultados.apply(lambda x: x['categorias'])

# Deixar apenas as colunas relevantes
df = df[['id','message_id','session_id','student_query', 'timestamp', 'gatilhos_encontrados', 'categorias_encontradas', 'NF', 'ESTADO', 'SEXO']]

# Salvar o DataFrame com os resultados
df.to_csv('../data/temp/chats_edu_gatilhos.csv', index=False, encoding='utf-8')


# Filtrar mensagens com gatilhos
df_com_gatilhos = df[df['gatilhos_encontrados'].apply(len) > 0]

# Análise dos resultados
print(f"\nEstatísticas:")
print(f"Total de mensagens analisadas: {len(df)}")
print(f"Mensagens com gatilhos metacognitivos: {len(df_com_gatilhos)}")

# Contagem por categoria
print("\nDistribuição por categoria:")
categoria_counts = defaultdict(int)
for categorias in df_com_gatilhos['categorias_encontradas']:
    for categoria in categorias:
        categoria_counts[categoria] += 1

for categoria, count in sorted(categoria_counts.items(), key=lambda x: x[1], reverse=True):
    percentual = (count / len(df)) * 100
    print(f"{categoria}: {count} mensagens ({percentual:.1f}%)")

# Mostrar exemplos de mensagens com gatilhos
print("\nExemplos de mensagens com gatilhos (primeiros 10):")
for idx, row in df_com_gatilhos.head(10).iterrows():
    print(f"\nMensagem: {row['student_query']}")
    print(f"Gatilhos encontrados: {row['gatilhos_encontrados']}")
    print(f"Categorias: {row['categorias_encontradas']}")

# Análise dos gatilhos mais frequentes
print("\nGatilhos mais frequentes:")
contagem_gatilhos = defaultdict(int)
for gatilhos in df_com_gatilhos['gatilhos_encontrados']:
    for gatilho in gatilhos:
        contagem_gatilhos[gatilho] += 1

print("\nTop 20 gatilhos mais frequentes:")
for gatilho, count in sorted(contagem_gatilhos.items(), key=lambda x: x[1], reverse=True)[:20]:
    percentual = (count / len(df)) * 100
    print(f"{gatilho}: {count} ocorrências ({percentual:.1f}%)")






Carregando dados...
Contando mensagens por sessão...
Processando mensagens...

Estatísticas:
Total de mensagens analisadas: 699
Mensagens com gatilhos metacognitivos: 203

Distribuição por categoria:
planejamento: 113 mensagens (16.2%)
reflexao: 61 mensagens (8.7%)
reconhecimento_erro: 43 mensagens (6.2%)
monitoramento: 41 mensagens (5.9%)
descoberta: 19 mensagens (2.7%)

Exemplos de mensagens com gatilhos (primeiros 10):

Mensagem: El concepto integracion se creo luego de que se conformara el concepto
Gatilhos encontrados: ['luego', 'forma']
Categorias: ['planejamento']

Mensagem: Dos amigos deciden regalar a su profesora un collar que tiene un valor de 7500 si uno de ellos aporta el doble que el otro y sabiendo que el menor aporte fue x entonces la ecuacion algebraica que representa tal situacion es
Gatilhos encontrados: ['decid']
Categorias: ['planejamento']

Mensagem: El perimetro de un triangulo isosceles es de 180cm cada uno de los lados iguales es 30 cm mayor que la base que val